# Solutions · 08 · Liquid-liquid extraction

Worked solutions to the exercises of [notebook 08](../notebooks/08_liquid_liquid_extraction.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import datasets
from engsep import extraction as ex
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
d = pd.read_csv(datasets.path("ternary_acetic_water_ether.csv"), comment="#")
tern = ex.Ternary(np.c_[d.xA_raffinate, d.xS_raffinate], np.c_[d.yA_extract, d.yS_extract], np.c_[d.xA_raffinate, d.yA_extract])

## Exercise 1

Recover 98 % of the acetic acid from the 30 % feed with an immiscible solvent of K = 3.0 (a better solvent than ether). Find S_min and the stages at 1.5 S_min, and compare with the ether case from the Hunter-Nash construction.

In [3]:
K, x_in, x_out = 3.0, 0.30, 0.006
S_min = ex.minimum_solvent_immiscible(K, x_in, x_out)
N = ex.countercurrent_stages_immiscible(K, 1.5 * S_min, x_in, x_out)
print(f"K = 3: S_min/F = {S_min:.3f}, at 1.5 S_min (S/F = {1.5*S_min:.2f}): {N:.1f} stages")
hn = ex.hunter_nash(tern, 100.0, (0.30, 0.0), 250.0, (0.0, 1.0), 0.02)
print(f"ether (K about 0.3-0.7, mutual solubility): S/F = 2.5 and {hn['stages']} stages for a raffinate of 0.02")

K = 3: S_min/F = 0.327, at 1.5 S_min (S/F = 0.49): 7.3 stages
ether (K about 0.3-0.7, mutual solubility): S/F = 2.5 and 8 stages for a raffinate of 0.02


A solvent with a distribution coefficient of 3 does the job with a fifth of the solvent-to-feed ratio and
about the same number of stages as ether, whose coefficient is below 1 over most of the range - and far
less solvent to recover afterwards. Solvent selection - distribution
coefficient, selectivity, immiscibility, and ease of recovery - decides an extraction before the stages are
counted.

## Exercise 2

Cross-current extraction of the 30 % acid feed with ether: mix the feed with 100 kg/h of ether, separate, and repeat with fresh ether three times. Use the tie-line data (lever rule each time) and compare the final raffinate with the countercurrent result at the same total solvent.

In [4]:
F, xF = 100.0, np.array([0.30, 0.0])
R, xR = F, xF.copy()
total_S = 0.0
for stage in range(1, 4):
    M, xM = ex.mixing_point(R, xR, 100.0, (0.0, 1.0))
    # split M along the tie line through it: find the raffinate a such that M lies on the tie line raffinate(a)-extract_in_equilibrium(a)
    from scipy import optimize
    def off_line(a):
        r, e = tern.raffinate(a), tern.extract_in_equilibrium(a)
        return (e[0] - r[0]) * (xM[1] - r[1]) - (e[1] - r[1]) * (xM[0] - r[0])
    a = optimize.brentq(off_line, 1e-4, xR[0])
    r, e = tern.raffinate(a), tern.extract_in_equilibrium(a)
    frac_E = ex.lever_split(xM, e, r)
    E, R = M * frac_E, M * (1 - frac_E)
    xR = r
    total_S += 100.0
    print(f"stage {stage}: raffinate {R:.1f} kg/h at {xR[0]:.3f} acid; extract {E:.1f} kg/h at {e[0]:.3f}")
cc = ex.hunter_nash(tern, 100.0, (0.30, 0.0), total_S, (0.0, 1.0), 0.02)
print(f"\ncountercurrent with the same total solvent ({total_S:.0f} kg/h): {cc['stages']} stages reach 0.02; cross-current in 3 stages reached {xR[0]:.3f}")

stage 1: raffinate 88.4 kg/h at 0.220 acid; extract 111.6 kg/h at 0.095
stage 2: raffinate 78.2 kg/h at 0.160 acid; extract 110.2 kg/h at 0.063
stage 3: raffinate 71.8 kg/h at 0.114 acid; extract 106.5 kg/h at 0.040

countercurrent with the same total solvent (300 kg/h): 6 stages reach 0.02; cross-current in 3 stages reached 0.114


Three cross-current stages with 100 kg/h of fresh ether each take the raffinate to about 12 % acid; the same
300 kg/h used countercurrently reaches 2 % in a few stages. The lever rule on the tie line through each
mixing point is the whole cross-current calculation; the countercurrent one needs the operating point.

## Exercise 3

**Implement it yourself:** write the Kremser form of countercurrent extraction from the stage balances (x_n = (x_{n-1} + E y_{n+1}/K)/(1+E)...) as an iterative stage-by-stage calculation and confirm the closed form for E = 1.5, N = 4.

In [5]:
E, N, K, x_in = 1.5, 4, 2.0, 0.10
# stage balances: R x_{n-1} + S y_{n+1} = R x_n + S y_n with y_n = K x_n; iterate the profile
x = np.full(N, x_in / 2)
for _ in range(500):
    x_new = x.copy()
    for n in range(N):
        x_prev = x_in if n == 0 else x[n - 1]
        y_next = 0.0 if n == N - 1 else K * x[n + 1]
        x_new[n] = (x_prev + E * y_next / K) / (1 + E)
    if np.max(np.abs(x_new - x)) < 1e-14:
        break
    x = x_new
print(f"stage-by-stage raffinate: x_N = {x[-1]:.6f}; Kremser closed form: {ex.countercurrent_immiscible(K, E / K, x_in, N)['x_out']:.6f}")

stage-by-stage raffinate: x_N = 0.007583; Kremser closed form: 0.007583


Iterating the stage balances to convergence reproduces the Kremser closed form to six digits: the closed
form is the solution of that linear system, and the iteration is what one would code for a nonlinear
distribution coefficient.